To store our data files, we create a temporary directory

# Experiment: Fine-Tuning von TinyBERT im FlowTransformer-Framework

**Autor:** Nikolai Steffan

**Kontext:** Studienarbeit Kapitel 6.2.1

**Status:** ❌ Experiment fehlgeschlagen (Architektonische Inkompatibilität)

---

## 1. Einleitung und Zielsetzung

Dieses Notebook dokumentiert den initialen Versuch, ein vortrainiertes **TinyBERT-Modell** mithilfe des **FlowTransformer-Frameworks** für die Intrusion Detection zu fine-tunen.

**Ziel:** Integration eines kompakten Sprachmodells (TinyBERT) in die bestehende Pipeline des Frameworks, um die Vorteile der *RecordLevelEmbed*-Input-Codierung mit der Leistungsfähigkeit von BERT zu kombinieren.

**Erwartetes Ergebnis:** Ein trainierbares Keras-Modell, das sequenzielle Netzwerkflüsse klassifiziert.
**Tatsächliches Ergebnis:** Wie in **Kapitel 6.2.1** der Studienarbeit analysiert, deckt dieses Experiment eine fundamentale architektonische Inkompatibilität zwischen der Embedding-Logik des Frameworks und den Input-Erwartungen von Standard-BERT-Modellen auf.

## 2. Setup und Initialisierung

Importieren der notwendigen Module aus dem `flow_transformer` Framework sowie der `TinyBERT` Implementierung. Ein Demonstrationsordner wird erstellt, um die Datensätze und Cache-Dateien zu speichern.

In [ ]:
import os

from implementations.classification_heads import LastTokenClassificationHead
from tiny_bert import TinyBERT
from framework.dataset_specification import DatasetSpecification
from framework.flow_transformer_parameters import FlowTransformerParameters
from framework.flow_transformer import FlowTransformer
from implementations.input_encodings import RecordLevelEmbed
from implementations.pre_processings import StandardPreProcessing
from framework.enumerations import EvaluationDatasetSampling
from IPython.display import display
import zipfile

demonstration_folder = "demonstration"

if not os.path.exists(demonstration_folder):
    os.mkdir(demonstration_folder)

/opt/anaconda3/envs/flow-transformer-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Datenbasis: UNSW-NB15

Wir verwenden den **UNSW-NB15** Datensatz als Grundlage. Im folgenden Schritt wird der Datensatz entpackt und für die Verarbeitung durch das Framework bereitgestellt. Der Datensatz kann [hier](https://staff.itee.uq.edu.au/marius/NIDS_datasets/) heruntergeladen werden

In [ ]:
csv_path = os.path.join(demonstration_folder, "dataset.csv")
zip_path = os.path.join(demonstration_folder, "NF-UNSW-NB15-v2.csv.zip") # Assuming this is the downloaded zip file name

if not os.path.exists(csv_path):
    # This part is for extraction, assuming the zip is already downloaded
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        # The internal path might vary, adjust if necessary
        internal_zip_path = "NF-UNSW-NB15-v2.csv" 
        zip_ref.extract(internal_zip_path, demonstration_folder)
        os.rename(os.path.join(demonstration_folder, internal_zip_path), csv_path)

print(f"Dataset is available at {csv_path}, size = {os.path.getsize(csv_path):,}")

Dataset is available at demonstration/dataset.csv, size = 577,360,958


## 4. Definition des Datenschemas

Das FlowTransformer-Framework benötigt eine strikte `DatasetSpecification`. Hier definieren wir:
* **Numerische Features:** Werden normalisiert.
* **Kategorische Features:** Werden für das Embedding vorbereitet.
* **Zielvariable:** "Attack" (binäre Klassifikation gegen "Benign").

In [ ]:
flow_format = DatasetSpecification(
        include_fields=['NUM_PKTS_UP_TO_128_BYTES', 'SRC_TO_DST_SECOND_BYTES', 'OUT_PKTS', 'OUT_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'DST_TO_SRC_AVG_THROUGHPUT', 'DURATION_IN', 'L4_SRC_PORT', 'ICMP_TYPE', 'PROTOCOL', 'SERVER_TCP_FLAGS', 'IN_PKTS', 'NUM_PKTS_512_TO_1024_BYTES', 'CLIENT_TCP_FLAGS', 'TCP_WIN_MAX_IN', 'NUM_PKTS_256_TO_512_BYTES', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'LONGEST_FLOW_PKT', 'L4_DST_PORT', 'MIN_TTL', 'DST_TO_SRC_SECOND_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'DURATION_OUT', 'FLOW_DURATION_MILLISECONDS', 'TCP_FLAGS', 'MAX_TTL', 'SRC_TO_DST_AVG_THROUGHPUT', 'ICMP_IPV4_TYPE', 'MAX_IP_PKT_LEN', 'RETRANSMITTED_OUT_BYTES', 'IN_BYTES', 'RETRANSMITTED_IN_BYTES', 'TCP_WIN_MAX_OUT', 'L7_PROTO', 'RETRANSMITTED_OUT_PKTS', 'RETRANSMITTED_IN_PKTS'],
        categorical_fields=['CLIENT_TCP_FLAGS', 'L4_SRC_PORT', 'TCP_FLAGS', 'ICMP_IPV4_TYPE', 'ICMP_TYPE', 'PROTOCOL', 'SERVER_TCP_FLAGS', 'L4_DST_PORT', 'L7_PROTO'],
        class_column="Attack",
        benign_label="Benign"
    )

## 5. Konstruktion der Pipeline

Hier wird die Architektur des Modells zusammengesetzt. Dies ist der kritische Teil des Experiments:

1.  **Input Encoding (`RecordLevelEmbed`):** Wandelt tabellarische Daten in dichte Vektoren (Embeddings) um.
2.  **Sequential Model (`TinyBERT`):** Das vortrainierte BERT-Modell soll die Sequenz dieser Embeddings verarbeiten.
3.  **Classification Head:** Klassifiziert basierend auf dem letzten Token.

⚠️ **Hinweis zur Architektur:** Das Framework berechnet Embeddings *bevor* sie an das Modell übergeben werden. Standard-BERT-Modelle erwarten jedoch meist Integer-IDs (`input_ids`), um ihre eigenen Embeddings zu generieren.

In [ ]:
pre_processing = StandardPreProcessing(n_categorical_levels=32)
encoding = RecordLevelEmbed(128)
transformer = TinyBERT()
classification_head = LastTokenClassificationHead()

# Enable fine-tuning on the BERT model's weights
transformer.set_trainable(True)

# Define the transformer
ft = FlowTransformer(pre_processing=pre_processing,
                     input_encoding=encoding,
                     sequential_model=transformer,
                     classification_head=classification_head,
                     params=FlowTransformerParameters(window_size=8, mlp_layer_sizes=[128], mlp_dropout=0.1))

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.decoder.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'bert.embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the 

## 6. Pre-Processing und Caching

Laden und Vorverarbeiten der Daten gemäß der Spezifikation. Das Framework nutzt ein Caching-System (`.feather` Dateien), um diesen Schritt bei wiederholter Ausführung zu beschleunigen.

In [ ]:
df = ft.load_dataset("UNSW-NB15",
                csv_path,
                specification=flow_format,
                evaluation_dataset_sampling=EvaluationDatasetSampling.LastRows,
                evaluation_percent=0.1,
                cache_path=demonstration_folder)

display(df.iloc[:500])

Using cache file path: demonstration/UNSW-NB15_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_5EjmvToFWKee8t20u0dFpVzNu4s0.feather
Reading directly from cache demonstration/UNSW-NB15_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_5EjmvToFWKee8t20u0dFpVzNu4s0.feather...


,SHORTEST_FLOW_PKT,DURATION_OUT,SRC_TO_DST_SECOND_BYTES,MIN_TTL,FLOW_DURATION_MILLISECONDS,MIN_IP_PKT_LEN,RETRANSMITTED_OUT_PKTS,IN_PKTS,NUM_PKTS_UP_TO_128_BYTES,DST_TO_SRC_AVG_THROUGHPUT,...,L7_PROTO_23,L7_PROTO_24,L7_PROTO_25,L7_PROTO_26,L7_PROTO_27,L7_PROTO_28,L7_PROTO_29,L7_PROTO_30,L7_PROTO_31,L7_PROTO_32
0,0.524437,0.000000,0.536183,0.625000,0.093873,0.659772,0.000000,0.069932,0.195001,0.770031,...,False,False,False,False,False,False,False,False,False,False
1,0.440913,0.496811,0.272973,0.625000,0.497056,0.608608,0.225897,0.336187,0.430769,0.657980,...,False,False,False,False,False,False,False,False,False,False
2,0.440913,0.667177,0.644755,0.625000,0.667200,0.608608,0.613318,0.552099,0.686549,0.847971,...,False,False,False,False,False,False,False,False,False,False
3,0.524437,0.000000,0.618112,0.625000,0.059228,0.659772,0.000000,0.069932,0.195001,0.793919,...,False,False,False,False,False,False,False,False,False,False
4,0.498267,0.000000,0.606954,0.625000,0.059228,0.642234,0.000000,0.069932,0.195001,0.788370,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.440913,0.343962,0.357933,0.625000,0.343955,0.608608,0.160932,0.258779,0.446948,0.702388,...,False,False,False,False,False,False,False,False,False,False
496,0.417031,0.443256,0.070595,0.999294,0.475800,0.596579,0.080466,0.110839,0.335929,0.516722,...,False,False,False,False,False,False,False,False,False,False
497,0.440913,0.429285,0.283114,0.625000,0.429277,0.608608,0.208002,0.297065,0.478736,0.663319,...,False,False,False,False,False,False,False,False,False,False
498,0.440913,0.440835,0.287095,0.625000,0.440827,0.608608,0.225897,0.316341,0.496074,0.665534,...,False,False,False,False,False,False,False,False,False,False


## 7. Modell-Kompilierung

Erstellen des Keras-Modells (`ft.build_model()`). Hier werden die Komponenten (Encoding -> Transformer -> Head) graphisch verbunden.

* **Optimizer:** Adam mit niedriger Lernrate (5e-5) für Fine-Tuning.
* **Loss:** Binary Crossentropy.

In [ ]:
import keras

m = ft.build_model()
m.summary()

# For fine-tuning, it is best practice to use a lower learning rate.
m.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-5), loss='binary_crossentropy', metrics=['binary_accuracy'], jit_compile=True)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_SHORTEST_FLO… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_DURATION_OUT  │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_SRC_TO_DST_S… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MIN_TTL       │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_FLOW_DURATIO… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MIN_IP_PKT_L… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_IN_PKTS       │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_NUM_PKTS_UP_… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_DST_TO_SRC_A… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MAX_TTL       │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_SRC_TO_DST_A… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_TCP_WIN_MAX_… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_LONGEST_FLOW… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_OUT_BYTES     │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_OUT_PKTS      │ (None, 8, 1)      │          0 │ -               

 Total params: 50,305 (196.50 KB)

 Trainable params: 50,305 (196.50 KB)

 Non-trainable params: 0 (0.00 B)

## 8. Training und Evaluation (Fehleranalyse)

Start des Trainingsprozesses.

❌ **Erwarteter Fehler:**
An dieser Stelle tritt der in der Arbeit beschriebene Fehler auf. Das `RecordLevelEmbed` Layer übergibt `Float`-Embeddings an das TinyBERT-Modell. Die `tf-keras` Kompatibilitätsschicht oder das Modell selbst können diese Inputs nicht korrekt zuordnen, da die interne Logik von Hugging Face Modellen primär auf `input_ids` (Integer) ausgelegt ist oder die Übergabe via `inputs_embeds` im Framework-Wrapper nicht korrekt propagiert wird.

Dies bestätigt die Notwendigkeit einer **benutzerdefinierten Implementierung** außerhalb dieses Frameworks, wie sie in **Kapitel 6.3** durchgeführt wurde.

In [ ]:
(train_results, eval_results, final_epoch) = ft.evaluate(m, batch_size=128, epochs=5, steps_per_epoch=64, early_stopping_patience=5)

print("Training results:")
display(train_results)

print("\nEvaluation results:")
display(eval_results)

Building eval dataset...
Splitting dataset to featurewise...
Evaluation dataset is built!
Positive samples in eval set: 0
Negative samples in eval set: 236542


ValueError: Exception encountered when calling Lambda.call().

[1mThe first argument to `Layer.call` must always be passed.[0m

Arguments received by Lambda.call():
  • inputs=tf.Tensor(shape=(128, 8, 128), dtype=float32)
  • mask=None
  • training=True